# Config

In [1]:
# Cellule 1: Changer de répertoire et charger le module
println("Re-loading NeuroDSL module after kernel fixes...")

# Important: on doit inclure depuis le dossier src/
include("src/NeuroDSL.jl")
using .NeuroDSL

println("--- Verification anti-duplication after reload ---")
checks = [
    (NeuroDSL.demand!,           "demand!",                    2),
    (NeuroDSL.backward_graph!,   "backward_graph!",            1),
    (NeuroDSL.execute_rule!,     "NeuroDSL.execute_rule!",     1),
    (NeuroDSL.accum_grad!,       "accum_grad!",                1),
]
all_ok = true
for (fn, name, max_m) in checks
    n = length(methods(fn))
    ok = n <= max_m
    global all_ok = all_ok && ok
    println(ok ? "✅" : "❌", " $name — $n méthode(s)")
end
if all_ok
    println("\n🚀 API chargée sans duplication.")
else
    error("❌ Duplication détectée !")
end

Re-loading NeuroDSL module after kernel fixes...
--- Verification anti-duplication after reload ---
✅ demand! — 1 méthode(s)
✅ backward_graph! — 1 méthode(s)
✅ NeuroDSL.execute_rule! — 1 méthode(s)
✅ accum_grad! — 1 méthode(s)

🚀 API chargée sans duplication.


# Test

In [ ]:
using NeuroDSL

In [ ]:
# Test 1
include("test/test_graph_api.jl")

In [ ]:
# Test 2
include("test/test_kernels.jl")

In [ ]:
# Test 3
include("test/test_backward.jl")

In [ ]:
# Test 4
include("test/test_layers.jl")

In [ ]:
println("=" ^ 50)

include("test/runtests.jl")

In [ ]:
include("test/test_runtime_optimizations.jl")

In [ ]:
# validation_mathematique.jl
using LinearAlgebra

# Logique extraite de vos fichiers (backward.jl et kernels.jl)
# On adapte simplement les types pour le Float64
function mse_loss_fwd_f64(out, target)
    return sum((out .- target).^2) / length(out)
end

function mse_loss_bwd_f64(out, target, dy)
    # Remplacé 2f0 par 2.0
    return (2.0 / length(out)) .* (out .- target) .* sum(dy)
end

function matmul_bwd_f64(A, B, dy, trans_b=false)
    if trans_b
        dA = dy * B
        dB = (A' * dy)'
    else
        dA = dy * B'
        dB = A' * dy
    end
    return dA, dB
end


In [ ]:
function test_gradient_logic()
    # Données aléatoires en Float64 (très précis)
    X = randn(Float64, 2, 4)
    W = randn(Float64, 4, 4)
    Target = zeros(Float64, 2, 4)
    
    # 1. Calcul Analytique (Votre logique)
    # Forward
    H = X # (simplifié pour l'exemple)
    Out = H * W' # (trans_b=true)
    Loss = mse_loss_fwd_f64(Out, Target)
    
    # Backward (Calcul de la dérivée manuelle)
    dy = 1.0 # Gradient de la loss par rapport à la sortie (initial)
    # Appliquez vos règles de dérivation ici
    dOut = mse_loss_bwd_f64(Out, Target, dy)
    _, dW = matmul_bwd_f64(H, W, dOut, true)
    
    # 2. Approximation Numérique (La "Vérité")
    eps = 1e-7
    W_plus = W + eps * ones(size(W)) # perturbation
    Out_plus = H * W_plus'
    Loss_plus = mse_loss_fwd_f64(Out_plus, Target)
    
    gradient_numérique = (Loss_plus - Loss) / eps
    
    # 3. Comparaison
    println("Erreur absolue : ", abs(sum(dW) - gradient_numérique))
end

In [ ]:
using LinearAlgebra

# 1. Vos fonctions (réécrites en Float64 pour le test)
mse_loss_bwd_f64(out, target, dy) = (2.0 ./ length(out)) .* (out .- target) .* sum(dy)

# Test unitaire simple
function test_gradients()
    # Données aléatoires en Float64
    out = randn(Float64, 4, 4)
    target = randn(Float64, 4, 4)
    dy = 1.0 

    # --- TEST DE MSE_LOSS ---
    # 1. Gradient analytique (le vôtre)
    grad_analytique = mse_loss_bwd_f64(out, target, dy)

    # 2. Gradient numérique
    eps = 1e-7
    # On perturbe 'out' pour voir l'effet sur la Loss
    out_plus = out .+ eps
    out_minus = out .- eps
    
    # Calcul de la fonction f(x)
    loss_plus = sum((out_plus .- target).^2) / length(out)
    loss_minus = sum((out_minus .- target).^2) / length(out)
    
    grad_numerique = (loss_plus - loss_minus) / (2 * eps)

    # 3. Comparaison
    diff = abs(sum(grad_analytique) - grad_numerique)
    println("Test MSE_LOSS :")
    println("  Analytique: ", sum(grad_analytique))
    println("  Numérique : ", grad_numerique)
    println("  Erreur    : ", diff)
    
    if diff < 1e-6
        println("  ✅ OK")
    else
        println("  ❌ Erreur élevée ! Vérifiez la formule.")
    end
end

test_gradients()

In [ ]:
function test_matmul_grad()
    A = randn(Float64, 2, 4)
    B = randn(Float64, 4, 3)
    dy = randn(Float64, 2, 3) # Gradient venant de la suite
    
    # Votre logique pour dA (dérivée par rapport à A)
    # Dans votre cas: dA = dy * B'
    dA_analytique = dy * B'
    
    # Approche numérique pour A
    eps = 1e-7
    # Perturbation de A uniquement
    A_plus = A .+ eps
    
    # Calcul Loss = sum( (A * B) * dy ) -> approximation du gradient
    loss_plus = sum((A_plus * B) .* dy)
    loss_minus = sum(((A .- eps) * B) .* dy)
    
    dA_numérique = (loss_plus - loss_minus) / (2 * eps)
    
    println("Test MATMUL dA :")
    println("  Analytique: ", sum(dA_analytique))
    println("  Numérique : ", dA_numérique)
    # Attention: ici on compare la somme des gradients
end

In [ ]:
using LinearAlgebra

# 1. Vos fonctions à tester (logique pure)
function matmul_bwd_check(A, B, dy, trans_b=false)
    # Copiez votre logique de backward.jl ici
    if trans_b
        dA = dy * B
        dB = (A' * dy)'
    else
        dA = dy * B'
        dB = A' * dy
    end
    return dA, dB
end

function test_matmul_grad()
    # Données
    A = randn(Float64, 2, 4)
    B = randn(Float64, 4, 3)
    dy = randn(Float64, 2, 3) # Le gradient en entrée (venant de la loss)
    
    # --- Vérification pour A ---
    eps = 1e-7
    # Gradient analytique (le vôtre)
    dA_analytique, _ = matmul_bwd_check(A, B, dy, false)
    
    # Gradient numérique (Finite Difference)
    # On perturbe A pour voir l'impact sur la sortie (A*B)
    dA_numerique = zeros(Float64, size(A))
    for i in 1:length(A)
        A_plus = copy(A); A_plus[i] += eps
        A_minus = copy(A); A_minus[i] -= eps
        
        # Loss fictive = somme de (Out .* dy) 
        # C'est la règle de la chaîne : dL/dA = (dOut/dA) * dy
        loss_plus = sum((A_plus * B) .* dy)
        loss_minus = sum((A_minus * B) .* dy)
        dA_numerique[i] = (loss_plus - loss_minus) / (2 * eps)
    end
    
    # --- Comparaison ---
    err_A = norm(dA_analytique - dA_numerique)
    println("Test Matmul dA : Erreur = $err_A")
    
    # Répétez pour B...
end

test_matmul_grad()

In [ ]:
function test_composition_chain()
    # 1. Setup
    X = randn(Float64, 2, 4)
    W = randn(Float64, 4, 4)
    Target = randn(Float64, 2, 4)
    
    # --- Analytique (Votre moteur) ---
    # Forward
    Out = X * W'
    Loss = sum((Out .- Target).^2) / length(Out)
    
    # Backward manuel (Règle de la chaîne)
    dy = (2.0 / length(Out)) .* (Out .- Target) # Gradient de la MSE
    dW = (X' * dy)' # Gradient du matmul
    
    # --- Numérique (La Vérité) ---
    eps = 1e-7
    W_plus = copy(W); W_plus[1,1] += eps
    Out_plus = X * W_plus'
    Loss_plus = sum((Out_plus .- Target).^2) / length(Out)
    
    dW_numérique = (Loss_plus - Loss) / eps
    
    # Comparaison
    diff = abs(dW[1,1] - dW_numérique)
    println("Test composition (MSE o Matmul) : Erreur = $diff")
end

In [ ]:
using LinearAlgebra

function test_composition_chain()
    # 1. Setup : Initialisation avec des valeurs déterministes pour la reproductibilité
    X = randn(Float64, 2, 4)
    W = randn(Float64, 4, 4)
    Target = randn(Float64, 2, 4)
    
    # --- Analytique (Votre moteur) ---
    # Forward pass
    Out = X * W'
    Loss = sum((Out .- Target).^2) / length(Out)
    
    # Backward manuel (Règle de la chaîne)
    # 1. Gradient de la loss par rapport à la sortie (dLoss/dOut)
    dy = (2.0 / length(Out)) .* (Out .- Target) 
    
    # 2. Gradient de la loss par rapport à W (dLoss/dW)
    # Si Out = X * W', alors dLoss/dW = (X' * dy)'
    dW = (X' * dy)' 
    
    # --- Numérique (La Vérité) ---
    # On vérifie uniquement le poids en [1,1]
    eps = 1e-7
    W_plus = copy(W)
    W_plus[1,1] += eps
    
    Out_plus = X * W_plus'
    Loss_plus = sum((Out_plus .- Target).^2) / length(Out)
    
    # Différence finie
    dW_numérique = (Loss_plus - Loss) / eps
    
    # --- Comparaison ---
    diff = abs(dW[1,1] - dW_numérique)
    
    println("--- Test de la Chaîne (MSE o Matmul) ---")
    println("Gradient Analytique (votre code) : ", dW[1,1])
    println("Gradient Numérique (approximation) : ", dW_numérique)
    println("Erreur absolue : ", diff)
    
    if diff < 1e-6
        println("✅ SUCCÈS : La règle de la chaîne est validée mathématiquement.")
    else
        println("❌ ÉCHEC : La propagation du gradient est incorrecte.")
    end
end

test_composition_chain()

# ADAMW

In [ ]:
using .NeuroDSL
using .Backend
using CUDA

# 1. PATCH UNIVERSEL : On force le graphe à TOUT mémoriser, sans exception.
function NeuroDSL.accum_grad!(nd::NeuroDSL.GraphNode, g_val)
    g_val === nothing && return
    if nd.gradient === nothing
        nd.gradient = similar(g_val)
        nd.gradient .= g_val
    else
        nd.gradient .+= g_val
    end
end

function test_integration_bypass()
    println("--- Démarrage du test (Bypass AdamW) ---")
    g = NeuroDSL.NeuroGraph(namespace=:simple_model, device=Backend.CPUDevice())
    
    NeuroDSL.set!(g, :w, Backend.ones32(g.device, 1, 1); is_param=true)
    NeuroDSL.set!(g, :x, Backend.ones32(g.device, 1, 1); is_param=false)
    NeuroDSL.set!(g, :target, Backend.ones32(g.device, 1, 1) .* 5.0f0; is_param=false)

    NeuroDSL.@addrules g :simple_model begin
        y = mul(w, x)
        loss = mse_loss(y, target)
    end

    w_node = NeuroDSL.node(g, :w; namespace=:simple_model)
    y_node = NeuroDSL.node(g, :y; namespace=:simple_model) # On récupère y pour enquêter
    
    # --- BOUCLE D'ENTRAÎNEMENT ---
    for t in 1:200
        NeuroDSL.zero_grads!(g)
        ctx_store = Dict{Symbol, Any}()
        
        # Forward pass
        NeuroDSL.demand!(g, :loss; ctx_store=ctx_store, namespace=:simple_model)
        
        # Backward pass
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx_store, namespace=:simple_model)
        
        # DEBUG IMMÉDIAT (Avant toute modification)
        if t == 1
            println("--- Debug Tour 1 (Post-Backward) ---")
            println("Gradient Y (doit être -8.0) : ", y_node.gradient === nothing ? "NOTHING" : Array(y_node.gradient)[1])
            println("Gradient W (doit être -8.0) : ", w_node.gradient === nothing ? "NOTHING" : Array(w_node.gradient)[1])
        end

        # OPTIMISATION MANUELLE (SGD basique - On ignore adamw_step!)
        if w_node.gradient !== nothing
            # w = w - (learning_rate * gradient)
            w_node.value .-= 0.05f0 .* w_node.gradient 
        end
        
        if t % 20 == 0
            val_w = Array(w_node.value)[1]
            println("Iteration $t | w: $(round(val_w, digits=4))")
        end
    end
    
    final_w = Array(w_node.value)[1]
    println(final_w > 4.5 ? "✅ SUCCÈS : $final_w" : "❌ ÉCHEC : $final_w")
end

test_integration_bypass()

In [ ]:
using .NeuroDSL
using .Backend
using CUDA

# 2. Initialisation des variables avec Float32 explicite
n = 5
W  = rand(Float32, n)            # Vos poids
dW = fill(Float32(-8.0), n)      # Gradient constant pour le test
m1 = zeros(Float32, n)           # Moment 1
m2 = zeros(Float32, n)           # Moment 2

# Paramètres
lr    = Float32(0.01)
b1    = Float32(0.9)
b2    = Float32(0.999)
eps_v = Float32(1e-8)
t     = Int32(1)
clip  = Float32(1.0)
wd    = Float32(0.01)

# Changement du nom ici : utilisez 'my_device' au lieu de 'device'
my_device = Backend.CPUDevice() 

# 3. Fonction de diagnostic
function run_adamw_test(dev, W, dW, m1, m2, lr, b1, b2, eps_v, t, clip, wd)
    println("--- Début du test AdamW ---")
    
    # Appel à votre fonction en utilisant 'dev'
    adamw_step!(dev, W, dW, m1, m2, lr, b1, b2, eps_v, t, clip, wd)
    
    println("\nValeurs après mise à jour :")
    println("W : ", W)
    
    # Vérification
    if all(dW .== Float32(0.0))
        println("✅ SUCCÈS : Le gradient a été remis à zéro.")
    else
        println("❌ ERREUR : Le gradient n'a pas été remis à zéro.")
    end
end

# 4. Exécution avec le nouveau nom
run_adamw_test(my_device, W, dW, m1, m2, lr, b1, b2, eps_v, t, clip, wd)